# 2일차 4교시 · 교차 검증

1. 한 번의 분할 결과가 얼마나 **흔들리는지** 확인합니다.
2. KFold와 StratifiedKFold의 차이를 봅니다.
3. `cross_validate`로 여러 지표를 **평균 ± 표준편차**로 요약합니다.
4. 교차 검증에서도 **Pipeline**이 누수를 막는다는 것을 확인합니다.

그래프 글자는 Colab에서 한글이 깨지지 않도록 영어로 적었습니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, cross_validate, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

X, y = load_breast_cancer(return_X_y=True)

## 1. 한 번의 분할은 운이다

분할 방법(`random_state`)만 바꿔 가며 결정 트리의 테스트 정확도를 20번 측정합니다.

In [ ]:
scores = []
for seed in range(20):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=seed, stratify=y)
    scores.append(DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_tr, y_tr).score(X_te, y_te))
print(f"최저 {min(scores):.3f}, 최고 {max(scores):.3f}, 평균 {np.mean(scores):.3f}")

plt.figure(figsize=(6, 3))
plt.plot(range(20), scores, marker="o")
plt.xlabel("split random_state")
plt.ylabel("test accuracy")
plt.title("Same model, different splits")
plt.show()

같은 모델인데 분할에 따라 정확도가 달라집니다. 숫자 하나로 모델을 판단하면 위험한 이유입니다.

## 2. KFold vs StratifiedKFold

데이터를 **정렬된 상태**로 두고 나누면 차이가 뚜렷합니다. 각 폴드의 검증 세트에 악성(0)이 몇 %인지 봅니다.

In [ ]:
order = np.argsort(y)                      # 라벨 순으로 정렬 (악성이 앞쪽)
X_sorted, y_sorted = X[order], y[order]
rows = []
for name, cv in [("KFold (섞지 않음)", KFold(5)), ("StratifiedKFold", StratifiedKFold(5))]:
    for i, (_, val_idx) in enumerate(cv.split(X_sorted, y_sorted)):
        rows.append({"방법": name, "폴드": i + 1,
                     "검증 세트 악성 비율": round((y_sorted[val_idx] == 0).mean(), 2)})
pd.DataFrame(rows).pivot(index="폴드", columns="방법", values="검증 세트 악성 비율")

원본 악성 비율은 약 0.37입니다. 섞지 않은 KFold는 어떤 폴드에는 악성만, 어떤 폴드에는 양성 종양만 들어갑니다. **분류 문제에는 StratifiedKFold**를 씁니다.

## 3. cross_validate로 여러 지표 요약하기

In [ ]:
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, pos_label=0),   # 악성 = 양성 클래스
    "recall": make_scorer(recall_score, pos_label=0),
    "f1": make_scorer(f1_score, pos_label=0),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
result = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring)

summary = pd.DataFrame({name: [result[f"test_{name}"].mean(), result[f"test_{name}"].std()] for name in scoring},
                       index=["평균", "표준편차"]).T.round(3)
summary

**보고 방법**: "재현율 0.956 ± 0.025 (5겹 층화 교차 검증)". 표준편차가 크면 모델이 데이터에 따라 불안정하다는 뜻입니다.

품질 게이트는 평균에 걸되, **평균 − 표준편차**도 기준을 넘는지 함께 보면 더 보수적입니다.

## 4. [선택] 교차 검증에서도 누수는 생긴다

1일차 5교시의 무작위 데이터 실험을 다시 봅니다. 특성 선택을 교차 검증 **밖에서** 하면 누수가 생깁니다.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

rng = np.random.RandomState(0)
X_noise = rng.normal(size=(100, 10000))
y_noise = rng.randint(0, 2, size=100)

leaky_X = SelectKBest(f_classif, k=20).fit_transform(X_noise, y_noise)
leaky = cross_val_score(LogisticRegression(max_iter=1000), leaky_X, y_noise, cv=5).mean()
safe = cross_val_score(make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=1000)),
                       X_noise, y_noise, cv=5).mean()
print(f"누수: {leaky:.2f}  /  Pipeline: {safe:.2f}  (정답은 0.5 근처)")

## 5. [선택] 연습

`n_splits`를 3, 5, 10으로 바꿔 보고 평균과 표준편차가 어떻게 변하는지 확인하세요.

In [ ]:
# 여기에 코드를 작성하세요